# Download data 

This code serves only to download the dataset used in the manuscript **'Tonic dopamine and biases in value learning linked through a biologically inspired reinforcement learning model'** by  Romero Pinto &  Uchida, 2023

The data was collected by Ju Tian and published originaly in  Tian, J. & Uchida, N. *Habenula Lesions Reveal that Multiple Mechanisms Underlie Dopamine Prediction Errors*. Neuron, 87(6), 1304–1316 [https://doi.org/10.1016/j.neuron.2015.08.028]

The public data repository is stored in the [**Open Science Framework**](https://osf.io/) repository
- The repository can be viewed [here](https://osf.io/cr5mv/?view_only=bd13a2d2de1947699b56ce70610b0e9b)
- The project ID is **cr5mv**
  
*Notes:*
- The configuration file for fetching the dataset is `.osfcii.config` and shouldn't be modified. 
- The downloaded data will be stored by default in the `data` folder in this repository. If needed to store elsewhere, modify the following:
  - `config_['output']` field below 
  - `g_dir` attribute in the `PathConfig` class (in `utils.path.py`)

In [ ]:
from osfclient import cli
from utils.data import *

config_=cli.config_from_file()
config_['output'] = 'data'

The function `clone_py` allows to specify which subfolder to download from the OSF project. Options are:
- `data_folder='raw_data'`: 
  <pre>
  </pre>  
  - Downloads the Habenula lesion data and biophysical model simulations for single neurons. 
  - Needed to run the notebooks: `data_analysis_LHb_data`,`fit_rl_models_LHb_data`,`drl_from_biophysical_simulations`,`rl_simulations_from_data`
<pre>
</pre>  
- `data_folder='rl_simulations'`:
  <pre>
  </pre>  
  - Downloads the RL simulations that are the output of `rl_sumulations_from_data`. These files are provided so that the user doesn't need to re-run the RL simulations (time consuming) and plot them directly. 
  - Needed to run the notebook:`plot_rl_simulations_from_data`
<pre>
</pre>  
- `data_folder='analysis'`: 
  <pre>
  </pre>
  - Downloads the fitted asymmetric scaling factors both from single neuron firing rates and from the biophysical simulation. These are computed in `data_analysis_LHb_data` and `drl_from_biophysical_simulations` respectively, but are given to the user for simplicity.
  - Needed to run the notebook:`plot_rl_simulations_from_data`




## Raw data format

Downloaded with: `data_folder='raw_data'`

These files contain the dataset used by Romero Pinto & Uchida, 2023.  The data is saved in a dictionary in a `.pkl` format for each single neuron. It contains two main datasets:

#### Habenula lesion dataset
Each file contains the electrophysiological recordings  made in an optotagged VTA dopamine neuron together with the task events of the session in which the neuron was recorded. Taken from Tian & Uchida 2015.

**`unit['data']`: Habenula lesion dataset**

- `unit['data']['TrialTypes']` : Index of trial type per trial  (Size: N trials)
- `unit['data']['TrialNames']` : Name of trial type per index (Size: N trialypes )

- `unit['data']['responses']`: **Responses from the habenula lesion dataset**

  - `unit['data']['responses']['lick']` : Timestamp of licks (Size: N licks per session)
  - `unit['data']['responses']['spike']`: Timestamp of spikes (Size: N spikes per session)

- `unit['data']['events']`: **Events in session from the habenula lesion dataset**

  - `unit['data']['events']['odorOn']`: Timestamp of odor onset per trial (Size: N trials)
  - `unit['data']['events']['odorOff']`: Timestamp of odor offset per trial (Size: N trials)
  - `unit['data']['events']['airpuffOn']`: Timestamp of airpuff onset per trial (Size: N trials)
  - `unit['data']['events']['rewardOn']`: Timestamp of reward onset per trial (Size: N trials)
  - `unit['data']['events']['trialStart']`: Timestamp of trialstart per trial (Size: N trials)
  - `unit['data']['events']['odorID']`: Index of odor per trial (Size:  N trials)

#### Biophysical simulation results

Simulation results from the biophysical simulations based on the data from the same recorded dopamine neuron. 

**`unit['simulation_results']`: Simulation results from the biophysical simulations based on data**

  - `unit['simulation_results']['da_conc']` : DA concentrarion per trial (Size: Timestamps per trial x N trials)
  - `unit['simulation_results']['d1_occ']` : D1 occupancy per trial (Size: Timestamps per trial x N trials)
  - `unit['simulation_results']['d2_occ']` : D2 occupancy per trial (Size: Timestamps per trial x N trials)
  - `unit['simulation_results']['input_fr']` : Input firing rate per trial (Size: Timestamps per trial x N trials)
  - `unit['simulation_results']['time_ax']` : time axis per trial (Size: Timestamps per trial x N trials)


In [ ]:
clone_py(config_, data_folder='raw_data')

## RL simulations format

Downloaded with: `data_folder='rl_simulations'`

Each file contains an RL simulation. Name syntax is as follows: "results_sim_type_TYPE_eps_EPS_GROUP_iteration_N" where:
- **TYPE**: type of simulation  {`sampling_`, `nosampling_nodistr_`, `nosampling_nodistr_`}  
  - *Note*: Look at notebook `rl_simulations_from_data` for detailed definitions on the simulation types
- **EPS**: $\beta$ parameter which is the decay factor in the models' update rule 
- **GROUP**: experimental group {lesion, control}
- **N**: iteration index. Simulation were run several times to get confidence intervals.

The file contains a dictionary with the following fields:

**`sim['agent']`:** contains information about the RL agent and algorithm employed in the simulation
- `sim['agent']['taus']`: single cell asymmetric scaling factor ($\tau_i$ parameter in the TD error equation below)
- `sim['agent']['eta']`: asymmetric scaling factor derived from receptor sensitivities  ($\eta$  parameter in the update equation below)
- `sim['agent']['epsilon']`: decay factor for the update equaton ($\beta$  parameter in the update equation below)
- `sim['agent']['base_alpha']`: additional step size for the updates ($\alpha$  parameter in the update equation below)
- `sim['agent']['do_sampling']`: logical indicating whether to perform sampling from the distribution in the TD error computation (only necesary in distributional RL)
- `sim['agent']['gamma']`: discounting factor  ($\gamma $ parameter in the TD error equation below)

$
\delta_{i,t} = r_t + \gamma \cdot \hat{z}(s_{t+1}) - V_i(s_t)\\
\hat{\delta}_{i,t}  = \tau_i \cdot \delta_{i,t} ~~~... ~~~\text{if} ~ \delta_{i,t} >0 \\
\hat{\delta}_{i,t}  = (1-\tau_i) \cdot \delta_{i,t} ~~~... ~~~\text{if} ~ \delta_{i,t} \leq 0
$

$
P_i(s_t) \leftarrow P_i(s_t) + \alpha \cdot \eta \cdot |\hat{\delta}_i(t)| - \beta \cdot P_i(s_t) ~~~... ~~~\text{if} ~ \delta_{i,t} >0\\
N_i(s_t) \leftarrow N_i(s_t) + \alpha \cdot (1-\eta) \cdot |\hat{\delta}_i(t)| - \beta \cdot N_i(s_t) ~~~... ~~~\text{if} ~ \delta_{i,t} \leq 0 \\
\hat{V}_i(s_t) = P_i(s_t) - N_i(s_t)
$  

**`sim['results']`:** contains the relevant variables  in the RL simulations
- `sim['results']['distribution_trials_full']`: distribution of value predictors ($V_i$) for each neuron, trial type, time bin and trial (Size: N cells x N trial types x N time steps x N trials)
- `sim['results']['distribution_g_trials_full']`: distribution of $P_i$ for each neuron, trial type, time bin and trial (Size: N cells x N trial types x N time steps x N trials)
- `sim['results']['distribution_ng_trials_full']`:  distribution of $N_i$ for each neuron, trial type, time bin and trial (Size: N cells x N trial types x N time steps x N trials)
- `sim['results']['delta_trials_full']`:  distribution of $\delta_i$ for each neuron, trial type, time bin and trial (Size: N cells x N trial types x N time steps x N trials)
- `sim['results']['distribution']`: distribution of  value predictors ($V_i$) for each neuron at convergence for each timebin (Size: N cells x N timebins across trial types)
- `sim['results']['distribution_g']`: distribution of $P_i$ for each neuron at convergence for each timebin (Size: N cells x N timebins across trial types) 
- `sim['results']['distribution_ng']`:  distribution of $N_i$ for each neuron at convergence for each timebin (Size: N cells x N timebins across trial types)
- `sim['results']['samp_distribution']`:  distribution of samples from the distribution of value predictors at convergence for each timebin (Size: N cells x N timebins across trial types)
- `sim['results']['deltas']`:  distribution of  $\delta_i$ for each neuron at convergence for each timebin (Size: N cells x N timebins across trial types)
  
**`sim['task']`**: contains the task configuration for the RL simnulation (based on the Pavlovian task of Tian & Uchida, 2015)
- `sim['task']['probs']`: probability of reward delivery for each trial type and each reward type (List of 4 elements for each trial type)
- `sim['task']['mags']`: reward magnitude for the corresponding probability in 'probs' above (List of 4 elements for each trial type)
- `sim['task']['cs_id']`: ID of cue type for each trial and reward type (List of 4 elements for each trial type)
- `sim['task']['str_cs']`: String corresponding to the name of each trial type (List of 4 elements for each trial type)
- `sim['task']['n_trials']`: Number of trials for the simulation
- `sim['task']['cue_onset']`: ID of the timebin at which the cue onset occurs within a trial
- `sim['task']['cue_dur']`: Duration of the cue presentation in the simulation (Unit: timebin)
- `sim['task']['rew_onset']`: ID of the timebin at which the reward onset occurs within a trial
- `sim['task']['iti_dur']`: Duration of the inter-trial interval (ITI) in the simulation (Unit: timebin)
- `sim['task']['percent_per_cs']`: Percentage of trials for each trial type (List of 4 elements for each trial type)
- `sim['task']['nb_cs']`: Number of cue types
- `sim['task']['trial_length']`: Duration of full trial (Unit: timebin)
- `sim['task']['deliv_rew']`:  Delivered reward for each trial and reward type (List of 8 elements)
- `sim['task']['deliv_cs']`: Delivered cue for each trial and reward type (List of 8 elements)
- `sim['task']['us_vector']`: Vector of rewards delivered for each trial in the simulation (Size: N trials)
- `sim['task']['cs_vector']`: Vector of cues delivered for each trial in the simulation (Size: N trials)
- `sim['task']['id_states_per_cs']`: ID of the states that are present in each for each trial type (List of 4 elements)


In [ ]:
clone_py(config_, data_folder='rl_simulations')

## Analysis format

Downloaded with: `data_folder='analysis'`

Each file contains a different set of parameters:

`taus_rec_occ_vs_da_level_population.npz`
- numpy array containing the asymmetric scaling factor derived from receptor sensitivities at a population level (Size: N groups x N iterations)

`drl_metrics_from_data.npz`
- dictionary containing the asymmetric scaling factors derived from single neuron firing rates for each group


In [ ]:
clone_py(config_, data_folder='analysis')

## Drug experiments
Downloaded with : `data_folder='drug_experiments'`

In [14]:
clone_py(config_, data_folder='drug_experiments')

100%|██████████| 561M/561M [00:33<00:00, 16.6Mbytes/s]
100%|██████████| 561M/561M [00:47<00:00, 11.9Mbytes/s]
100%|██████████| 561M/561M [00:47<00:00, 11.7Mbytes/s]
100%|██████████| 561M/561M [00:46<00:00, 12.2Mbytes/s]
100%|██████████| 561M/561M [00:45<00:00, 12.3Mbytes/s]
100%|██████████| 561M/561M [00:50<00:00, 11.1Mbytes/s]
100%|██████████| 561M/561M [00:47<00:00, 11.9Mbytes/s]
100%|██████████| 561M/561M [00:50<00:00, 11.1Mbytes/s]
100%|██████████| 561M/561M [00:48<00:00, 11.5Mbytes/s]
100%|██████████| 561M/561M [00:46<00:00, 12.1Mbytes/s]
100%|██████████| 561M/561M [00:50<00:00, 11.1Mbytes/s]
100%|██████████| 561M/561M [00:48<00:00, 11.6Mbytes/s]
100%|██████████| 561M/561M [00:46<00:00, 12.0Mbytes/s]
100%|██████████| 561M/561M [00:44<00:00, 12.6Mbytes/s]
100%|██████████| 561M/561M [00:46<00:00, 12.1Mbytes/s]
100%|██████████| 561M/561M [00:49<00:00, 11.3Mbytes/s]
100%|██████████| 561M/561M [00:49<00:00, 11.4Mbytes/s]
100%|██████████| 561M/561M [00:51<00:00, 10.8Mbytes/s]
100%|█████

ProtocolError: ('Connection broken: IncompleteRead(97426637 bytes read, 463949742 more expected)', IncompleteRead(97426637 bytes read, 463949742 more expected))